## Notebook 2 – Training and comparing surrogate models

In this notebook we **continue from Notebook 1** and work with the *same* IEA3.4 training data:

1. **Repeat / reproduce the data split**  
   - We load the chosen operational regime (e.g. `Normal`, `Start`, `Shut`, `Idling`),  
   - Apply the same splitting logic as before (random / stratified on target / stratified on (WS, TI, target), or K-fold),  
   - Keep a clear separation between **train**, **validation**, and the **independent test set** (test is only used in Notebook 3).

2. **Train different surrogate models for one selected target channel**  
   For the same inputs (`Wind Speed`, `Turbulence Intensity`, possibly more) and a single target (e.g. `PowAc_mean`, `TBFA_DEL4`, `BRFW_DEL10`), we train:
   - **Neural networks (MLP)**  
     - with a small **manual grid search** over hyperparameters,  
     - and with **Optuna** for automatic hyperparameter search.
   - **XGBoost (gradient boosted trees)**  
     - with a manual **grid search**,  
     - and with **Optuna**.
   - **Gaussian Process Regression (GPR)**  
     - comparing several kernels (RBF, Matérn, and simple combinations).

For each model we:
- Choose **input/output scalers** (e.g. standard, min–max, log),
- Select a **primary error metric** (MAE, MSE, RMSE, MAPE),
- Train on the **training** set (or via K-fold on the full factorial data),
- Evaluate on the **validation** set and report ME, MAE, MSE, RMSE, MAPE, and R² in **original physical units**.

### Educational focus

The goal is **not** to build the “perfect” surrogate, but to:
- See how **splitting strategy**, **scaling**, and **hyperparameters** affect model quality,
- Compare **different model families** (NN, trees, GPR) on the *same* load/power target,
- Prepare one or more trained models (saved to disk) that we can later:
  - reload in **Notebook 3**,
  - sweep over wind speed and turbulence intensity,
  - and analyze in detail using domain knowledge and richer error diagnostics.


In [1]:
#CELL 1  Imports and global configuration

import os
import pickle

import numpy as np
import pandas as pd

from sklearn.model_selection import train_test_split, KFold, StratifiedKFold
from sklearn.gaussian_process import GaussianProcessRegressor
from sklearn.gaussian_process.kernels import (
    Sum,
    Product,
    RBF,
    Matern,
    WhiteKernel,
    ConstantKernel
)
from sklearn.preprocessing import StandardScaler, MinMaxScaler, RobustScaler
from sklearn.neural_network import MLPRegressor
from sklearn.metrics import r2_score

import xgboost as xgb
import optuna
from itertools import product  

# Directory paths
DATA_DIR = "Data"
STUDENT_DIR = "student"

os.makedirs(STUDENT_DIR, exist_ok=True)

# Regime and target selection (students can change these)
SELECTED_REGIME = "Normal"   # "Normal", "Idling", "Shut", "Start"
TARGET_COL = "MSBMZ_loc_DEL4"    # e.g. "PowAc_mean", "TBFA_DEL4", "TBSS_DEL4", "BRFW_DEL10","BREW_DEL10", "MSBMY_loc_DEL4","MSBMZ_loc_DEL4", "TTTOR_DEL4"

# Inputs to the surrogates
INPUT_COLS = ["Wind Speed", "Turbulence Intensity"]

# Split configuration (must match Notebook 1 if you want identical splits)
SPLIT_METHOD = "stratified_target"  # "random", "stratified_target", "stratified_joint"

TRAIN_FRACTION = 0.8
RANDOM_STATE = 42

# Binning parameters for stratified splits
TARGET_STRAT_NBINS = 6
WSP_STRAT_NBINS = 3
TI_STRAT_NBINS = 3
TARGET_JOINT_NBINS = 3


In [2]:
#CELL 2: Load development and test data

train_file = os.path.join(DATA_DIR, f"{SELECTED_REGIME}_training_data.csv")
test_file  = os.path.join(DATA_DIR, f"{SELECTED_REGIME}_test_data.csv")

if not os.path.exists(train_file) or not os.path.exists(test_file):
    raise FileNotFoundError(f"Missing files for {SELECTED_REGIME}: {train_file} or {test_file}")

# In this notebook:
# - df_full is the DEVELOPMENT DATA used for training/validation/K-fold tuning.
# - df_test is the UNTOUCHED TEST SET and should only be used in Notebook 3.
df_full = pd.read_csv(train_file)
df_test = pd.read_csv(test_file)

print(f"Loaded regime: {SELECTED_REGIME}")
print("  Development data:", df_full.shape)
print("  Test set:        ", df_test.shape, "(kept untouched in this notebook)")

Loaded regime: Normal
  Development data: (329, 92)
  Test set:         (94, 92) (kept untouched in this notebook)


In [3]:
#CELL 3: Split helper functions (same logic as Notebook 1, reduced methods)

def make_quantile_bins(series, nbins):
    """
    Make quantile-based bins for a 1D series, returning bin labels (ints).
    """
    nbins = min(nbins, series.nunique())
    if nbins <= 1:
        return np.zeros(len(series), dtype=int)

    quantiles = np.linspace(0, 1, nbins + 1)
    edges = series.quantile(quantiles).values
    edges = np.unique(edges)
    if len(edges) - 1 < nbins:
        nbins = len(edges) - 1
    labels = pd.cut(series, bins=edges, labels=False, include_lowest=True)
    return labels.astype(int)


def split_random(df, train_fraction, random_state):
    df_train, df_val = train_test_split(
        df, train_size=train_fraction, random_state=random_state, shuffle=True
    )
    return df_train.copy(), df_val.copy()


def split_stratified_target(df, target_col, train_fraction, nbins, random_state):
    y = df[target_col]
    bins = make_quantile_bins(y, nbins)
    df_train, df_val = train_test_split(
        df,
        train_size=train_fraction,
        random_state=random_state,
        shuffle=True,
        stratify=bins,
    )
    return df_train.copy(), df_val.copy()


def build_joint_strata_labels(df, wsp_col, ti_col, target_col,
                              n_wsp_bins, n_ti_bins, n_target_bins):
    wsp_bins = make_quantile_bins(df[wsp_col], n_wsp_bins)
    ti_bins = make_quantile_bins(df[ti_col], n_ti_bins)
    target_bins = make_quantile_bins(df[target_col], n_target_bins)

    labels = [
        f"w{iw}_t{it}_y{iy}"
        for iw, it, iy in zip(wsp_bins, ti_bins, target_bins)
    ]
    return np.array(labels)


def split_stratified_joint(df, wsp_col, ti_col, target_col,
                           train_fraction, n_wsp_bins, n_ti_bins,
                           n_target_bins, random_state):
    labels = build_joint_strata_labels(
        df,
        wsp_col=wsp_col,
        ti_col=ti_col,
        target_col=target_col,
        n_wsp_bins=n_wsp_bins,
        n_ti_bins=n_ti_bins,
        n_target_bins=n_target_bins,
    )
    df_train, df_val = train_test_split(
        df,
        train_size=train_fraction,
        random_state=random_state,
        shuffle=True,
        stratify=labels,
    )
    return df_train.copy(), df_val.copy()

# Apply chosen split

wsp_col = "Wind Speed"
ti_col = "Turbulence Intensity"

if SPLIT_METHOD == "random":
    df_train, df_val = split_random(
        df_full,
        train_fraction=TRAIN_FRACTION,
        random_state=RANDOM_STATE,
    )
elif SPLIT_METHOD == "stratified_target":
    df_train, df_val = split_stratified_target(
        df_full,
        target_col=TARGET_COL,
        train_fraction=TRAIN_FRACTION,
        nbins=TARGET_STRAT_NBINS,
        random_state=RANDOM_STATE,
    )
elif SPLIT_METHOD == "stratified_joint":
    df_train, df_val = split_stratified_joint(
        df_full,
        wsp_col=wsp_col,
        ti_col=ti_col,
        target_col=TARGET_COL,
        train_fraction=TRAIN_FRACTION,
        n_wsp_bins=WSP_STRAT_NBINS,
        n_ti_bins=TI_STRAT_NBINS,
        n_target_bins=TARGET_JOINT_NBINS,
        random_state=RANDOM_STATE,
    )
else:
    raise ValueError(f"Unknown SPLIT_METHOD: {SPLIT_METHOD}")

print("\nFinal split sizes:")
print("  Train:", df_train.shape)
print("  Val  :", df_val.shape)
print("  Test :", df_test.shape)


# %% Build X/y arrays

def build_xy(df, input_cols, target_col):
    X = df[input_cols].values.astype(float)
    y = df[target_col].values.astype(float)
    return X, y

X_train, y_train = build_xy(df_train, INPUT_COLS, TARGET_COL)
X_val,   y_val   = build_xy(df_val,   INPUT_COLS, TARGET_COL)
X_test,  y_test  = build_xy(df_test,  INPUT_COLS, TARGET_COL)  # used in Notebook 3
X_full,  y_full  = build_xy(df_full,  INPUT_COLS, TARGET_COL)  # full development data

print("\nShapes:")
print("  X_train:", X_train.shape, " y_train:", y_train.shape,  "(traing set)")
print("  X_val  :", X_val.shape,   " y_val  :", y_val.shape,  "(validation set)")
print("  X_test :", X_test.shape,  " y_test :", y_test.shape, "(kept untouched here)")
print("  X_full :", X_full.shape,  " y_full :", y_full.shape, "(full development data)")



Final split sizes:
  Train: (263, 92)
  Val  : (66, 92)
  Test : (94, 92)

Shapes:
  X_train: (263, 2)  y_train: (263,) (traing set)
  X_val  : (66, 2)  y_val  : (66,) (validation set)
  X_test : (94, 2)  y_test : (94,) (kept untouched here)
  X_full : (329, 2)  y_full : (329,) (full development data)


In [4]:
#CELL 4: Scaler classes, error metrics and helpers

class LogScaler:
    """
    Simple log-transform scaler.

    transform: z = log(x + shift)
    inverse_transform: x = exp(z) - shift

    shift is chosen so that x + shift > 0 for all training samples.
    Works with 1D or 2D arrays and preserves input shape.
    """
    def __init__(self):
        self.shift_ = None

    def fit(self, x):
        x = np.asarray(x, dtype=float)
        x_flat = x.ravel()
        xmin = np.min(x_flat)
        if xmin <= 0:
            self.shift_ = -xmin + 1e-6
        else:
            self.shift_ = 0.0
        return self

    def transform(self, x):
        x = np.asarray(x, dtype=float)
        return np.log(x + self.shift_)

    def inverse_transform(self, z):
        z = np.asarray(z, dtype=float)
        return np.exp(z) - self.shift_

    def fit_transform(self, x):
        self.fit(x)
        return self.transform(x)



def create_scaler(name):
    """
    Factory for scalers.

    name: "standard", "minmax", "robust", "log", or "none".
    """
    if name is None or name.lower() == "none":
        return None
    name = name.lower()
    if name == "standard":
        return StandardScaler()
    elif name == "minmax":
        return MinMaxScaler()
    elif name == "robust":
        return RobustScaler()
    elif name == "log":
        return LogScaler()
    else:
        raise ValueError(f"Unknown scaler name: {name}")


def fit_transform_scaler(scaler, x_train, x_other_list=None):
    """
    Fit scaler on x_train and transform x_train and optional other arrays.

    Returns transformed x_train and list of transformed others.
    If scaler is None, returns inputs unchanged.
    """
    if scaler is None:
        if x_other_list is None:
            return x_train, None
        else:
            return x_train, x_other_list

    scaler.fit(x_train)
    x_train_scaled = scaler.transform(x_train)
    x_others_scaled = None
    if x_other_list is not None:
        x_others_scaled = [scaler.transform(arr) for arr in x_other_list]
    return x_train_scaled, x_others_scaled


# Metrics helpers

def mean_error(y_true, y_pred):
    y_true = np.asarray(y_true, dtype=float)
    y_pred = np.asarray(y_pred, dtype=float)
    return float(np.mean(y_true - y_pred))


def mae(y_true, y_pred):
    y_true = np.asarray(y_true, dtype=float)
    y_pred = np.asarray(y_pred, dtype=float)
    return float(np.mean(np.abs(y_true - y_pred)))


def mse(y_true, y_pred):
    y_true = np.asarray(y_true, dtype=float)
    y_pred = np.asarray(y_pred, dtype=float)
    diff = y_true - y_pred
    return float(np.mean(diff * diff))


def rmse(y_true, y_pred):
    return float(np.sqrt(mse(y_true, y_pred)))


def mape(y_true, y_pred, eps=1e-8):
    y_true = np.asarray(y_true, dtype=float)
    y_pred = np.asarray(y_pred, dtype=float)
    denom = np.maximum(np.abs(y_true), eps)
    return float(np.mean(np.abs((y_true - y_pred) / denom)) * 100.0)


def r2(y_true, y_pred):
    y_true = np.asarray(y_true, dtype=float)
    y_pred = np.asarray(y_pred, dtype=float)
    return float(r2_score(y_true, y_pred))


ERROR_FUNCTIONS = {
    "mae": mae,
    "mse": mse,
    "rmse": rmse,
    "mape": mape,
}

def compute_all_metrics(y_true, y_pred):
    """
    Compute all standard regression metrics in original physical units.

    These metrics are used for validation/CV reporting during model selection.
    Final test metrics are computed later in Notebook 3.
    """
    return {
        "ME": mean_error(y_true, y_pred),
        "MAE": mae(y_true, y_pred),
        "MSE": mse(y_true, y_pred),
        "RMSE": rmse(y_true, y_pred),
        "MAPE": mape(y_true, y_pred),
        "R2": r2(y_true, y_pred),
    }

#  Helper: stratified KFold on continuous target

def build_stratified_bins_for_cv(y, nbins):
    """
    Build bin labels for y (1D) for use with StratifiedKFold in regression.
    """
    series = pd.Series(y)
    bins = make_quantile_bins(series, nbins)
    return bins


# Helpers for saving/loading model bundles

def build_student_model_path(regime, target, model_type, run_tag):
    """
    Build a filename for a student model bundle.
    """
    safe_regime = str(regime).replace(" ", "")
    safe_target = str(target).replace(" ", "")
    safe_type = str(model_type).replace(" ", "")
    safe_tag = str(run_tag).replace(" ", "")
    filename = f"{safe_regime}__{safe_target}__{safe_type}__{safe_tag}.pkl"
    return os.path.join(STUDENT_DIR, filename)


def save_model_bundle(path, model, x_scaler, y_scaler, meta_dict):
    """
    Save {model, x_scaler, y_scaler, meta} dict to a pickle file.
    """
    bundle = {
        "model": model,
        "x_scaler": x_scaler,
        "y_scaler": y_scaler,
        "meta": meta_dict,
    }
    with open(path, "wb") as f:
        pickle.dump(bundle, f)
    print(f"Saved model bundle to: {path}")


### Neural network surrogate – grid search

In this section we train a **fully connected neural network (MLP)** as a load surrogate.

What happens in this cell:

- We use the **train/validation split** from Notebook 1 (or from earlier in this notebook).
- Inputs: the columns in `INPUT_COLS` (typically `Wind Speed` and `Turbulence Intensity`).
- Target: the single channel selected in `TARGET_COL` (e.g. `PowAc_mean`, `TBFA_DEL4`, etc.).
- We apply **X and Y scalers** chosen by `X_SCALER_NAME` and `Y_SCALER_NAME`
  (e.g. `standard`, `minmax`, `log`, or `none`).
- We perform a **manual grid search** over NN hyperparameters:
  - number of neurons / layers (`hidden_layer_sizes`),
  - activation function (`activation`),
  - regularization (`alpha`),
  - optional learning rate / max iterations.

For each hyperparameter combination:

- The model is trained on the **training set** (or via K-Fold on the full factorial data if `USE_KFOLD=True`).
- We evaluate on the **validation set** using a chosen error metric (`PRIMARY_METRIC_NAME`, e.g. `rmse`).
- All metrics (ME, MAE, MSE, RMSE, MAPE, R²) are computed in the **original physical units** of the target.

What to explore:

- Change the **scalers** (e.g. try `Y_SCALER_NAME = "log"` vs `"none"` for fatigue loads).
- Change the **grid** (fewer/more neurons, different activations) and see how validation metrics change.
- Compare a “small” NN vs a “large” NN in terms of R² and MAPE on the validation set.
- Observe if K-Fold (`USE_KFOLD=True`) gives more stable metrics than a single train/val split.


In [ ]:
# CELL 5: Evaluation function for NN configs

def evaluate_nn_config(params, use_kfold):
    """
    Evaluate one NN hyperparameter configuration.

    Model-selection logic
    ---------------------
    If use_kfold is False:
        - Fit candidate model on X_train, y_train.
        - Evaluate candidate model on X_val, y_val.

    If use_kfold is True:
        - Use X_full, y_full as development data.
        - Rotate validation folds using StratifiedKFold.
        - Report mean/std metrics over folds.

    Returns
    -------
    score : float
        Value of PRIMARY_METRIC_NAME used for selecting the best configuration.
    details : dict
        Validation/CV metrics and fold information.
    """

    if use_kfold:
        # Stratified KFold on the full development data
        y_bins = build_stratified_bins_for_cv(y_full, nbins=N_BINS_STRAT)
        skf = StratifiedKFold(
            n_splits=N_SPLITS_KFOLD,
            shuffle=True,
            random_state=RANDOM_STATE,
        )

        fold_metric_rows = []

        for fold_idx, (train_idx, val_idx) in enumerate(skf.split(X_full, y_bins), start=1):
            X_tr = X_full[train_idx]
            y_tr = y_full[train_idx]
            X_va = X_full[val_idx]
            y_va = y_full[val_idx]

            # Fresh scalers for each fold.
            # Fit scalers only on the training fold.
            x_scaler = create_scaler(X_SCALER_NAME)
            y_scaler = create_scaler(Y_SCALER_NAME)

            if x_scaler is not None:
                X_tr_scaled = x_scaler.fit_transform(X_tr)
                X_va_scaled = x_scaler.transform(X_va)
            else:
                X_tr_scaled = X_tr
                X_va_scaled = X_va

            if y_scaler is not None:
                y_tr_scaled = y_scaler.fit_transform(
                    y_tr.reshape(-1, 1)
                ).ravel()
            else:
                y_tr_scaled = y_tr

            # Candidate model
            model = MLPRegressor(
                hidden_layer_sizes=params["hidden_layer_sizes"],
                activation=params["activation"],
                alpha=params["alpha"],
                learning_rate_init=params["learning_rate_init"],
                max_iter=params["max_iter"],
                random_state=RANDOM_STATE,
            )

            model.fit(X_tr_scaled, y_tr_scaled)

            # Validation-fold prediction
            y_va_pred_scaled = model.predict(X_va_scaled)

            if y_scaler is not None:
                y_va_pred = y_scaler.inverse_transform(
                    y_va_pred_scaled.reshape(-1, 1)
                ).ravel()
            else:
                y_va_pred = y_va_pred_scaled

            # Metrics in original physical units
            fold_metrics = compute_all_metrics(y_va, y_va_pred)
            fold_metrics["fold"] = fold_idx
            fold_metric_rows.append(fold_metrics)

        df_folds = pd.DataFrame(fold_metric_rows)

        # Mean and std metrics across folds
        mean_metrics = {
            f"{metric}_mean": float(df_folds[metric].mean())
            for metric in ["ME", "MAE", "MSE", "RMSE", "MAPE", "R2"]
        }

        std_metrics = {
            f"{metric}_std": float(df_folds[metric].std())
            for metric in ["ME", "MAE", "MSE", "RMSE", "MAPE", "R2"]
        }

        # Primary score used for model selection
        score = mean_metrics[f"{PRIMARY_METRIC_NAME.upper()}_mean"]

        details = {
            "selection_type": "kfold_cv",
            "fold_metrics": df_folds,
        }
        details.update(mean_metrics)
        details.update(std_metrics)

        return score, details

    else:
        # Fixed validation strategy:
        # fit candidate on training data, evaluate candidate on validation data
        X_tr, y_tr = X_train, y_train
        X_va, y_va = X_val, y_val

        x_scaler = create_scaler(X_SCALER_NAME)
        y_scaler = create_scaler(Y_SCALER_NAME)

        # Fit scalers only on training data
        if x_scaler is not None:
            X_tr_scaled = x_scaler.fit_transform(X_tr)
            X_va_scaled = x_scaler.transform(X_va)
        else:
            X_tr_scaled = X_tr
            X_va_scaled = X_va

        if y_scaler is not None:
            y_tr_scaled = y_scaler.fit_transform(
                y_tr.reshape(-1, 1)
            ).ravel()
        else:
            y_tr_scaled = y_tr

        # Candidate model
        model = MLPRegressor(
            hidden_layer_sizes=params["hidden_layer_sizes"],
            activation=params["activation"],
            alpha=params["alpha"],
            learning_rate_init=params["learning_rate_init"],
            max_iter=params["max_iter"],
            random_state=RANDOM_STATE,
        )

        model.fit(X_tr_scaled, y_tr_scaled)

        # Validation prediction
        y_va_pred_scaled = model.predict(X_va_scaled)

        if y_scaler is not None:
            y_va_pred = y_scaler.inverse_transform(
                y_va_pred_scaled.reshape(-1, 1)
            ).ravel()
        else:
            y_va_pred = y_va_pred_scaled

        # Metrics in original physical units
        val_metrics = compute_all_metrics(y_va, y_va_pred)

        # Primary score used for model selection
        score = val_metrics[PRIMARY_METRIC_NAME.upper()]

        details = {
            "selection_type": "fixed_validation",
        }
        details.update(val_metrics)

        return score, details

In [ ]:
# CELL 6: NN with manual grid search

from itertools import product

# Choose scalers for inputs and target  
X_SCALER_NAME = "standard"   # "standard", "minmax", "robust", "log", "none"
Y_SCALER_NAME = "standard"       # "standard", "minmax", "robust", "log", "none" 

# Choose primary optimization metric for the grid search
PRIMARY_METRIC_NAME = "rmse"   # "mae", "mse", "rmse", "mape"

# KFold configuration for tuning
USE_KFOLD = True  #  If true, use Stratified KFold on full data; else train/val split
N_SPLITS_KFOLD = 4
N_BINS_STRAT = 6   # bins for stratified KFold on y

SAVE_MODEL = False
RUN_TAG = "my_model" # add a tag to identify your model bundle file

# Grid of hyperparameters for MLPRegressor
# This is a *superset* – you can comment out entries to reduce combinations in class.
NN_PARAM_GRID = {
    # 1-layer and 2-layer networks with moderate widths
    "hidden_layer_sizes": [
        # (32,),
        # (64,),
        (32, 32),
        # (64, 64),
        (64,64,64)
    ],
    # Common activations for regression
    "activation": [
        "relu",
        # "tanh",
        # "logistic",
    ],

    # L2 regularization strength
    "alpha": [
        1e-5,
        # 1e-4,
        # 1e-3,
        # 1e-2,
    ],
    # Initial learning rate (for 'adam' solver)
    "learning_rate_init": [
        # 1e-4,
        1e-3,
        # 3e-3,
        # 1e-2,
    ],
    # Max iterations to optimize weights
    "max_iter": [
        250,
    ],
}


print("\n=== NN grid search ===")
print("X scaler:", X_SCALER_NAME, " Y scaler:", Y_SCALER_NAME)
print("Primary metric:", PRIMARY_METRIC_NAME)
print("Use KFold:", USE_KFOLD)

metric_fn = ERROR_FUNCTIONS[PRIMARY_METRIC_NAME]



# ------------------------------------------------------------
# Run grid search: model selection using validation/CV only
# ------------------------------------------------------------

results = []
all_fold_metrics = []   # only used when USE_KFOLD=True

for candidate_id, values in enumerate(product(*NN_PARAM_GRID.values()), start=1):
    params = dict(zip(NN_PARAM_GRID.keys(), values))

    score, details = evaluate_nn_config(params, USE_KFOLD)

    row = {
        "candidate_id": candidate_id,
        "params": params,
        "selection_score": score,
        "primary_metric": PRIMARY_METRIC_NAME,
        "selection_type": details["selection_type"],
    }

    # Add all scalar metrics to the results table.
    # Do not add fold_metrics directly as a table column unless needed.
    for key, value in details.items():
        if key not in ["fold_metrics"]:
            row[key] = value

   # Keep fold metrics separately for visible inspection
    if USE_KFOLD:
        row["fold_metrics"] = details["fold_metrics"]

        df_candidate_folds = details["fold_metrics"].copy()
        df_candidate_folds.insert(0, "candidate_id", candidate_id)
        df_candidate_folds["params"] = str(params)

        all_fold_metrics.append(df_candidate_folds)


    # Always store the candidate result, both for fixed validation and K-fold
    results.append(row)

    print("\nParams:", params)
    print("Selection score", f"({PRIMARY_METRIC_NAME}) =", score)

# Collect results in DataFrame
df_results_nn = pd.DataFrame(results).sort_values(
    "selection_score",
    ascending=True,
)

print("\nNN grid-search validation/CV results")
print("Sorted by:", PRIMARY_METRIC_NAME)

display(
    df_results_nn.drop(columns=["fold_metrics"], errors="ignore")
)

# If K-fold was used, show all fold-by-fold metrics in a separate table.
if USE_KFOLD:
    df_nn_fold_metrics = pd.concat(all_fold_metrics, ignore_index=True)

    print("\nNN grid-search fold-by-fold validation metrics")
    print("One row per candidate and validation fold.")
    display(df_nn_fold_metrics)
else:
    df_nn_fold_metrics = None

# Pick best configuration
best_row = df_results_nn.iloc[0]
best_params = best_row["params"]

print("\nBest NN grid-search setup:")
print("  Primary metric:", PRIMARY_METRIC_NAME)
print("  Selection type:", best_row["selection_type"])
print("  Selection score:", best_row["selection_score"])
print("  Best params:", best_params)

# Optional: inspect fold metrics for the best model if K-fold was used
if USE_KFOLD:
    print("\nFold metrics for the best NN setup:")
    display(best_row["fold_metrics"])


# ------------------------------------------------------------
# Retrain final NN model on all development data
# ------------------------------------------------------------
# The validation/CV scores above were used only for model selection.
# After selecting the best hyperparameters, the final model is retrained
# on all development data: X_full, y_full.
#
# Final performance is evaluated later in Notebook 3 using the untouched test set.

X_tr_final, y_tr_final = X_full, y_full

x_scaler_final = create_scaler(X_SCALER_NAME)
y_scaler_final = create_scaler(Y_SCALER_NAME)

# Fit final scalers on all development data
if x_scaler_final is not None:
    X_tr_final_scaled = x_scaler_final.fit_transform(X_tr_final)
else:
    X_tr_final_scaled = X_tr_final

if y_scaler_final is not None:
    y_tr_final_scaled = y_scaler_final.fit_transform(
        y_tr_final.reshape(-1, 1)
    ).ravel()
else:
    y_tr_final_scaled = y_tr_final

best_nn_model = MLPRegressor(
    hidden_layer_sizes=best_params["hidden_layer_sizes"],
    activation=best_params["activation"],
    alpha=best_params["alpha"],
    learning_rate_init=best_params["learning_rate_init"],
    max_iter=best_params["max_iter"],
    random_state=RANDOM_STATE,
)

best_nn_model.fit(X_tr_final_scaled, y_tr_final_scaled)

print("\nFinal NN grid-search model retrained on all development data.")
print("Use Notebook 3 to evaluate this saved model on the untouched test set and compare with the reults from the validation.")

# Optional: save the model bundle
if SAVE_MODEL:
    model_type = "nn_grid"
    path = build_student_model_path(SELECTED_REGIME, TARGET_COL, model_type, RUN_TAG)
    meta = {
    "regime": SELECTED_REGIME,
    "target": TARGET_COL,
    "model_type": model_type,
    "input_cols": INPUT_COLS,
    "x_scaler_name": X_SCALER_NAME,
    "y_scaler_name": Y_SCALER_NAME,
    "primary_metric": PRIMARY_METRIC_NAME,
    "split_method": SPLIT_METHOD,
    "use_kfold": USE_KFOLD,
    "param_grid": NN_PARAM_GRID,
    "best_params": best_params,
    "selection_type": best_row["selection_type"],
    "selection_score": float(best_row["selection_score"]),
    "selection_metrics": {
        key: best_row[key]
        for key in df_results_nn.columns
        if key not in ["params", "fold_metrics"]
    },
    "final_training_data": "all_development_data_X_full_y_full",
}
    save_model_bundle(path, best_nn_model, x_scaler_final, y_scaler_final, meta)


### Neural network surrogate – Optuna hyperparameter search

In this section we train a **neural network surrogate** again, but now we use
**Optuna** to search the hyperparameter space instead of a fixed grid.

What happens in this cell:

- We keep the same data setup:
  - same inputs (`INPUT_COLS`),
  - same target (`TARGET_COL`),
  - same train/validation split and scalers (`X_SCALER_NAME`, `Y_SCALER_NAME`).
- Optuna samples NN hyperparameters (e.g. `hidden_layer_sizes`, `activation`, `alpha`, `max_iter`) from specified ranges.
- For each sampled configuration:
  - We call the same evaluation logic as in the grid search (optionally with **Stratified K-Fold** on the full factorial data if `USE_KFOLD=True`).
  - We compute the chosen primary metric (`PRIMARY_METRIC_NAME`, e.g. RMSE) on the **validation set**.
- Optuna keeps track of the best configuration and returns the **best hyperparameters**.
- We then retrain a final NN using these best settings and report the full set of metrics (ME, MAE, MSE, RMSE, MAPE, R²) on the validation set.

What to explore:

- Change `N_TRIALS` to see the effect of more/fewer Optuna trials.
- Modify the **search ranges** (e.g. allow deeper networks or stronger regularization) and see what Optuna prefers.
- Compare the **best Optuna model** against the **best grid-search model**:
  - Which one gives better RMSE / MAPE?
  - Are the chosen hyperparameters similar or very different?
  - How do the training times compare?


In [ ]:
# CELL 7: NN with Optuna hyperparameter search


# Choose scalers for inputs and target
# All options: "standard", "minmax", "robust", "log", "none"
X_SCALER_NAME = "standard"
Y_SCALER_NAME = "log"

# Choose primary optimization metric for Optuna
# Options: "mae", "mse", "rmse", "mape"
PRIMARY_METRIC_NAME = "rmse"

# KFold configuration for tuning (same interpretation as before)
USE_KFOLD = True 
N_SPLITS_KFOLD = 4
N_BINS_STRAT = 6   # bins for stratified KFold on y

# Optuna configuration
N_TRIALS = 5      # you can reduce this for speed
TIMEOUT = None     # or set seconds if you want a hard limit

# Defines a dictionary with hidden layer sizes and their optuna labels that can be chosen for hyperparameter tuning.
SIZE_MAP = {
    "32":       (32,),
    "64":       (64,),
    "32_32":    (32, 32),
    "64_64":    (64, 64),
    "64_64_64": (64, 64, 64),
}

SAVE_MODEL = True
RUN_TAG = "my_model"  # add a tag to identify your model bundle file

print("\n=== NN Optuna search ===")
print("X scaler:", X_SCALER_NAME, " Y scaler:", Y_SCALER_NAME)
print("Primary metric:", PRIMARY_METRIC_NAME)
print("Use KFold:", USE_KFOLD)
print("N_TRIALS:", N_TRIALS)

metric_fn = ERROR_FUNCTIONS[PRIMARY_METRIC_NAME]

# We reuse the StratifiedKFold configuration from before.
# evaluate_nn_config() is already defined in the previous cell and
# uses X_SCALER_NAME, Y_SCALER_NAME, metric_fn, USE_KFOLD, etc.


def objective(trial):
    """
    Optuna objective function.

    Samples NN hyperparameters, calls evaluate_nn_config, and returns
    the chosen error metric (to be minimized).
    """
    # Hyperparameter search space (you can tune/trim these ranges)
    # Choose hidden layer sizes from dictionary SIZE_MAP using the defined labels (see above)
    hidden_layer_sizes = SIZE_MAP[trial.suggest_categorical(
        "hidden_layer_sizes",
        [
            "32",
            "64",
            "32_32",
            "64_64",
            "64_64_64",
        ]
    )]
    activation = trial.suggest_categorical(
        "activation",
        ("relu", "tanh"),  # you can add "logistic", "identity" if desired
    )
    # L2 regularization (alpha), log scale
    alpha = trial.suggest_float("alpha", 1e-3, 1e-1, log=True)
    # Learning rate, log scale
    # learning_rate_init = trial.suggest_float("learning_rate_init", 1e-4, 1e-2, log=True)
    learning_rate_init = trial.suggest_categorical("learning_rate_init", (1e-4,1e-3, 1e-2))
    # Max iterations (integer range)
    # max_iter = trial.suggest_int("max_iter", 200, 500, step=100) # 
    max_iter = trial.suggest_categorical("max_iter", (200, 500)) 

    params = {
        "hidden_layer_sizes": hidden_layer_sizes,
        "activation": activation,
        "alpha": alpha,
        "learning_rate_init": learning_rate_init,
        "max_iter": max_iter,
    }

    # Reuse our existing evaluation logic
    score, details = evaluate_nn_config(params, USE_KFOLD)

    # You can log intermediate info if you want:
    trial.set_user_attr("details", details)

    return score


# Create and run the study
study = optuna.create_study(direction="minimize")
study.optimize(objective, n_trials=N_TRIALS, timeout=TIMEOUT, show_progress_bar=False,n_jobs=1)

# ------------------------------------------------------------
# Collect and display Optuna validation/CV results
# ------------------------------------------------------------
# Each Optuna trial is one candidate NN setup.
# The score used by Optuna is the validation/CV score from evaluate_nn_config().
# The untouched test set is not used here.

trial_rows = []
all_fold_metrics = []   # only used when USE_KFOLD=True

for trial in study.trials:
    # Skip failed or pruned trials, if any
    if trial.value is None:
        continue

    details = trial.user_attrs.get("details", {})

    row = {
        "trial_number": trial.number,
        "selection_score": trial.value,
        "primary_metric": PRIMARY_METRIC_NAME,
        "selection_type": details.get("selection_type", None),
    }

    # Add Optuna hyperparameters as separate columns
    for p_name, p_value in trial.params.items():
        row[p_name] = p_value

    # Add scalar validation/CV metrics
    for key, value in details.items():
        if key not in ["fold_metrics", "selection_type"]:
            row[key] = value

    # Keep fold metrics separately for visible inspection
    if USE_KFOLD and "fold_metrics" in details:
        df_trial_folds = details["fold_metrics"].copy()
        df_trial_folds.insert(0, "trial_number", trial.number)
        df_trial_folds["params"] = str(trial.params)

        all_fold_metrics.append(df_trial_folds)

    trial_rows.append(row)


df_trials_nn_optuna = pd.DataFrame(trial_rows).sort_values(
    "selection_score",
    ascending=True,
).reset_index(drop=True)

print("\nNN Optuna validation/CV results")
print("One row per Optuna trial.")
print("Sorted by:", PRIMARY_METRIC_NAME)
display(df_trials_nn_optuna)

# If K-fold was used, show all fold-by-fold metrics in a separate table.
if USE_KFOLD:
    df_nn_optuna_fold_metrics = pd.concat(all_fold_metrics, ignore_index=True)

    print("\nNN Optuna fold-by-fold validation metrics")
    print("One row per trial and validation fold.")
    display(df_nn_optuna_fold_metrics)
else:
    df_nn_optuna_fold_metrics = None


print("\nOptuna best value (", PRIMARY_METRIC_NAME, "):", study.best_value)
print("Optuna best params:", study.best_params)

best_params = study.best_params

print("\nBest NN Optuna setup:")
print("  Primary metric:", PRIMARY_METRIC_NAME)
print("  Selection score:", study.best_value)
print("  Best params:", best_params)


# ------------------------------------------------------------
# Retrain final NN model on all development data
# ------------------------------------------------------------
# The validation/CV scores above were used only for model selection.
# After selecting the best hyperparameters, the final model is retrained
# on all development data: X_full, y_full.
#
# Final performance is evaluated later in Notebook 3 using the untouched test set.

X_tr_final, y_tr_final = X_full, y_full

x_scaler_final = create_scaler(X_SCALER_NAME)
y_scaler_final = create_scaler(Y_SCALER_NAME)

# Fit final scalers on all development data
if x_scaler_final is not None:
    X_tr_final_scaled = x_scaler_final.fit_transform(X_tr_final)
else:
    X_tr_final_scaled = X_tr_final

if y_scaler_final is not None:
    y_tr_final_scaled = y_scaler_final.fit_transform(
        y_tr_final.reshape(-1, 1)
    ).ravel()
else:
    y_tr_final_scaled = y_tr_final

best_nn_model = MLPRegressor(
    hidden_layer_sizes=SIZE_MAP[best_params["hidden_layer_sizes"]],
    activation=best_params["activation"],
    alpha=best_params["alpha"],
    learning_rate_init=best_params["learning_rate_init"],
    max_iter=best_params["max_iter"],
    random_state=RANDOM_STATE,
)

best_nn_model.fit(X_tr_final_scaled, y_tr_final_scaled)

print("\nFinal NN Optuna model retrained on all development data.")
print("Use Notebook 3 to evaluate this saved model on the untouched test set and compare with the validation/CV results.")

# Optional: save the model bundle
if SAVE_MODEL:
    model_type = "nn_optuna"
    path = build_student_model_path(SELECTED_REGIME, TARGET_COL, model_type, RUN_TAG)
    meta = {
    "regime": SELECTED_REGIME,
    "target": TARGET_COL,
    "model_type": model_type,
    "input_cols": INPUT_COLS,
    "x_scaler_name": X_SCALER_NAME,
    "y_scaler_name": Y_SCALER_NAME,
    "primary_metric": PRIMARY_METRIC_NAME,
    "split_method": SPLIT_METHOD,
    "use_kfold": USE_KFOLD,
    "n_trials": N_TRIALS,
    "best_params": best_params,
    "best_value": study.best_value,
    "selection_type": df_trials_nn_optuna.iloc[0]["selection_type"],
    "selection_score": float(df_trials_nn_optuna.iloc[0]["selection_score"]),
    "selection_metrics": {
        key: df_trials_nn_optuna.iloc[0][key]
        for key in df_trials_nn_optuna.columns
        if key not in ["params"]
    },
    "final_training_data": "all_development_data_X_full_y_full",
    }
    save_model_bundle(path, best_nn_model, x_scaler_final, y_scaler_final, meta)


### XGBoost surrogate – grid search

Here we train a **gradient-boosted tree surrogate** (XGBoost) for the same regression problem.

What happens in this cell:

- Inputs and target are the same as before (`INPUT_COLS`, `TARGET_COL`).
- We apply the chosen **X and Y scalers** (`X_SCALER_NAME`, `Y_SCALER_NAME`).
- We perform a **manual grid search** over XGBoost hyperparameters:
  - `n_estimators`: number of trees,
  - `max_depth`: depth of each tree,
  - `learning_rate`: step size for boosting,
  - `subsample`, `colsample_bytree`: subsampling of rows/columns,
  - `reg_lambda`: L2 regularization.
- For each hyperparameter combination:
  - The model is trained on the training data (or via K-Fold on `df_full` if `USE_KFOLD=True`).
  - We compute the primary metric (`PRIMARY_METRIC_NAME`) on the **validation set**.
- We sort all configurations by the chosen metric and select the **best XGBoost model**.
- For the best model we compute and print ME, MAE, MSE, RMSE, MAPE, and R² on the validation set.

What to explore:

- Compare **tree-based** behaviour vs NN:
  - How do the metrics differ for the same target?
  - Does XGBoost struggle or perform better on some channels?
- Reduce or expand the **parameter grid** (e.g. fewer `n_estimators`, smaller `max_depth`) and observe the effect on validation performance and training time.
- Try different **Y scalers** (e.g. `log` for fatigue loads vs `none`) and see the impact on MAPE/RMSE.


In [ ]:
# CELL 8: XGBoost with manual grid search

# Choose scalers for inputs and target
# All options: "standard", "minmax", "robust", "log", "none"
X_SCALER_NAME = "standard"
Y_SCALER_NAME = "minmax"

# Choose primary optimization metric for the grid search
# Options: "mae", "mse", "rmse", "mape"
PRIMARY_METRIC_NAME = "rmse"
metric_fn = ERROR_FUNCTIONS[PRIMARY_METRIC_NAME]

# KFold configuration for tuning
USE_KFOLD = True
N_SPLITS_KFOLD = 5
N_BINS_STRAT = 6   # bins for stratified KFold on y

# Grid of hyperparameters for XGBRegressor.
XGB_PARAM_GRID = {
    "n_estimators": [200, 400],       # fewer vs more boosting rounds
    "max_depth": [3, 5],              # shallow vs moderately deep trees
    "learning_rate": [0.03, 0.1],     # low vs moderate learning rate
    "subsample": [0.7, 1.0],          # row sampling (stochastic vs full)
    "colsample_bytree": [0.7, 1.0],   # feature sampling
    "reg_lambda": [1.0],              # L2 regularization on
}

SAVE_MODEL = True
RUN_TAG = "my_model"

print("\n=== XGBoost grid search ===")
print("X scaler:", X_SCALER_NAME, " Y scaler:", Y_SCALER_NAME)
print("Primary metric:", PRIMARY_METRIC_NAME)
print("Use KFold:", USE_KFOLD)

def evaluate_xgb_config(params, use_kfold):
    """
    Evaluate one XGBoost hyperparameter configuration.

    Model-selection logic
    ---------------------
    If use_kfold is False:
        - Fit candidate model on X_train, y_train.
        - Evaluate candidate model on X_val, y_val.

    If use_kfold is True:
        - Use X_full, y_full as development data.
        - Rotate validation folds using StratifiedKFold.
        - Report mean/std metrics over folds.

    Returns
    -------
    score : float
        Value of PRIMARY_METRIC_NAME used for selecting the best configuration.
        Lower is better.
    details : dict
        Validation/CV metrics and fold information.
    """

    if use_kfold:
        # Stratified KFold on the full development data
        y_bins = build_stratified_bins_for_cv(y_full, nbins=N_BINS_STRAT)
        skf = StratifiedKFold(
            n_splits=N_SPLITS_KFOLD,
            shuffle=True,
            random_state=RANDOM_STATE,
        )

        fold_metric_rows = []

        for fold_idx, (train_idx, val_idx) in enumerate(skf.split(X_full, y_bins), start=1):
            X_tr = X_full[train_idx]
            y_tr = y_full[train_idx]
            X_va = X_full[val_idx]
            y_va = y_full[val_idx]

            # Create fresh scalers for each fold
            # Fit scalers only on the training fold
            x_scaler = create_scaler(X_SCALER_NAME)
            y_scaler = create_scaler(Y_SCALER_NAME)

            if x_scaler is not None:
                X_tr_scaled = x_scaler.fit_transform(X_tr)
                X_va_scaled = x_scaler.transform(X_va)
            else:
                X_tr_scaled = X_tr
                X_va_scaled = X_va

            if y_scaler is not None:
                y_tr_scaled = y_scaler.fit_transform(
                    y_tr.reshape(-1, 1)
                ).ravel()
            else:
                y_tr_scaled = y_tr

            model = xgb.XGBRegressor(
                n_estimators=params["n_estimators"],
                max_depth=params["max_depth"],
                learning_rate=params["learning_rate"],
                subsample=params["subsample"],
                colsample_bytree=params["colsample_bytree"],
                reg_lambda=params["reg_lambda"],
                objective="reg:squarederror",
                random_state=RANDOM_STATE,
            )

            model.fit(X_tr_scaled, y_tr_scaled)

            # Predict on validation fold
            y_va_pred_scaled = model.predict(X_va_scaled)

            if y_scaler is not None:
                y_va_pred = y_scaler.inverse_transform(
                    y_va_pred_scaled.reshape(-1, 1)
                ).ravel()
            else:
                y_va_pred = y_va_pred_scaled

            # Metrics in original physical units
            fold_metrics = compute_all_metrics(y_va, y_va_pred)
            fold_metrics["fold"] = fold_idx
            fold_metric_rows.append(fold_metrics)

        df_folds = pd.DataFrame(fold_metric_rows)

        # Mean and std metrics across folds
        mean_metrics = {
            f"{metric}_mean": float(df_folds[metric].mean())
            for metric in ["ME", "MAE", "MSE", "RMSE", "MAPE", "R2"]
        }

        std_metrics = {
            f"{metric}_std": float(df_folds[metric].std())
            for metric in ["ME", "MAE", "MSE", "RMSE", "MAPE", "R2"]
        }

        # Primary score used for model selection
        score = mean_metrics[f"{PRIMARY_METRIC_NAME.upper()}_mean"]

        details = {
            "selection_type": "kfold_cv",
            "fold_metrics": df_folds,
        }
        details.update(mean_metrics)
        details.update(std_metrics)

        return score, details

    else:
        # Fixed validation strategy:
        # fit candidate on training data, evaluate candidate on validation data
        X_tr, y_tr = X_train, y_train
        X_va, y_va = X_val, y_val

        x_scaler = create_scaler(X_SCALER_NAME)
        y_scaler = create_scaler(Y_SCALER_NAME)

        # Fit scalers only on training data
        if x_scaler is not None:
            X_tr_scaled = x_scaler.fit_transform(X_tr)
            X_va_scaled = x_scaler.transform(X_va)
        else:
            X_tr_scaled = X_tr
            X_va_scaled = X_va

        if y_scaler is not None:
            y_tr_scaled = y_scaler.fit_transform(
                y_tr.reshape(-1, 1)
            ).ravel()
        else:
            y_tr_scaled = y_tr

        model = xgb.XGBRegressor(
            n_estimators=params["n_estimators"],
            max_depth=params["max_depth"],
            learning_rate=params["learning_rate"],
            subsample=params["subsample"],
            colsample_bytree=params["colsample_bytree"],
            reg_lambda=params["reg_lambda"],
            objective="reg:squarederror",
            random_state=RANDOM_STATE,
        )

        model.fit(X_tr_scaled, y_tr_scaled)

        # Predict on validation set
        y_va_pred_scaled = model.predict(X_va_scaled)

        if y_scaler is not None:
            y_va_pred = y_scaler.inverse_transform(
                y_va_pred_scaled.reshape(-1, 1)
            ).ravel()
        else:
            y_va_pred = y_va_pred_scaled

        # Metrics in original physical units
        val_metrics = compute_all_metrics(y_va, y_va_pred)

        # Primary score used for model selection
        score = val_metrics[PRIMARY_METRIC_NAME.upper()]

        details = {
            "selection_type": "fixed_validation",
        }
        details.update(val_metrics)

        return score, details

# ------------------------------------------------------------
# Run grid search: model selection using validation/CV only
# ------------------------------------------------------------

results_xgb = []
all_fold_metrics_xgb = []   # only used when USE_KFOLD=True

for candidate_id, values in enumerate(product(*XGB_PARAM_GRID.values()), start=1):
    params = dict(zip(XGB_PARAM_GRID.keys(), values))

    score, details = evaluate_xgb_config(params, USE_KFOLD)

    row = {
        "candidate_id": candidate_id,
        "params": params,
        "selection_score": score,
        "primary_metric": PRIMARY_METRIC_NAME,
        "selection_type": details["selection_type"],
    }

    # Add all scalar metrics to the results table.
    # Do not add fold_metrics directly as a table column unless needed.
    for key, value in details.items():
        if key not in ["fold_metrics"]:
            row[key] = value

    # Keep fold metrics separately for visible inspection
    if USE_KFOLD:
        row["fold_metrics"] = details["fold_metrics"]

        df_candidate_folds = details["fold_metrics"].copy()
        df_candidate_folds.insert(0, "candidate_id", candidate_id)
        df_candidate_folds["params"] = str(params)

        all_fold_metrics_xgb.append(df_candidate_folds)

    results_xgb.append(row)

    print("\nParams:", params)
    print("Selection score", f"({PRIMARY_METRIC_NAME}) =", score)


# Collect results in DataFrame
df_results_xgb = pd.DataFrame(results_xgb).sort_values(
    "selection_score",
    ascending=True,
).reset_index(drop=True)

print("\nXGBoost grid-search validation/CV results")
print("Sorted by:", PRIMARY_METRIC_NAME)

display(
    df_results_xgb.drop(columns=["fold_metrics"], errors="ignore")
)

# If K-fold was used, show all fold-by-fold metrics in a separate table.
if USE_KFOLD:
    df_xgb_fold_metrics = pd.concat(all_fold_metrics_xgb, ignore_index=True)

    print("\nXGBoost grid-search fold-by-fold validation metrics")
    print("One row per candidate and validation fold.")
    display(df_xgb_fold_metrics)
else:
    df_xgb_fold_metrics = None


# Pick best configuration
best_row_xgb = df_results_xgb.iloc[0]
best_params_xgb = best_row_xgb["params"]

print("\nBest XGBoost grid-search setup:")
print("  Primary metric:", PRIMARY_METRIC_NAME)
print("  Selection type:", best_row_xgb["selection_type"])
print("  Selection score:", best_row_xgb["selection_score"])
print("  Best params:", best_params_xgb)


# ------------------------------------------------------------
# Retrain final XGBoost model on all development data
# ------------------------------------------------------------
# The validation/CV scores above were used only for model selection.
# After selecting the best hyperparameters, the final model is retrained
# on all development data: X_full, y_full.
#
# Final performance is evaluated later in Notebook 3 using the untouched test set.

X_tr_final, y_tr_final = X_full, y_full

x_scaler_final = create_scaler(X_SCALER_NAME)
y_scaler_final = create_scaler(Y_SCALER_NAME)

# Fit final scalers on all development data
if x_scaler_final is not None:
    X_tr_final_scaled = x_scaler_final.fit_transform(X_tr_final)
else:
    X_tr_final_scaled = X_tr_final

if y_scaler_final is not None:
    y_tr_final_scaled = y_scaler_final.fit_transform(
        y_tr_final.reshape(-1, 1)
    ).ravel()
else:
    y_tr_final_scaled = y_tr_final

best_xgb_model = xgb.XGBRegressor(
    n_estimators=best_params_xgb["n_estimators"],
    max_depth=best_params_xgb["max_depth"],
    learning_rate=best_params_xgb["learning_rate"],
    subsample=best_params_xgb["subsample"],
    colsample_bytree=best_params_xgb["colsample_bytree"],
    reg_lambda=best_params_xgb["reg_lambda"],
    objective="reg:squarederror",
    random_state=RANDOM_STATE,
)

best_xgb_model.fit(X_tr_final_scaled, y_tr_final_scaled)

print("\nFinal XGBoost grid-search model retrained on all development data.")
print("Use Notebook 3 to evaluate this saved model on the untouched test set and compare with the validation/CV results.")

# Optional: save the model bundle
if SAVE_MODEL:
    model_type = "xgb_grid"
    path = build_student_model_path(SELECTED_REGIME, TARGET_COL, model_type, RUN_TAG)
    meta = {
        "regime": SELECTED_REGIME,
        "target": TARGET_COL,
        "model_type": model_type,
        "input_cols": INPUT_COLS,
        "x_scaler_name": X_SCALER_NAME,
        "y_scaler_name": Y_SCALER_NAME,
        "primary_metric": PRIMARY_METRIC_NAME,
        "split_method": SPLIT_METHOD,
        "use_kfold": USE_KFOLD,
        "param_grid": XGB_PARAM_GRID,
        "best_params": best_params_xgb,
        "selection_type": best_row_xgb["selection_type"],
        "selection_score": float(best_row_xgb["selection_score"]),
        "selection_metrics": {
            key: best_row_xgb[key]
            for key in df_results_xgb.columns
            if key not in ["params", "fold_metrics"]
        },
        "final_training_data": "all_development_data_X_full_y_full",
    }

    save_model_bundle(path, best_xgb_model, x_scaler_final, y_scaler_final, meta)

### XGBoost surrogate – Optuna hyperparameter search

In this section we train an XGBoost surrogate again, this time using **Optuna**
to search the hyperparameter space instead of a fixed grid.

What happens in this cell:

- The data setup remains the same:
  - same train/validation data,
  - same inputs and target,
  - same scalers (`X_SCALER_NAME`, `Y_SCALER_NAME`).
- Optuna samples XGBoost hyperparameters:
  - `n_estimators`, `max_depth`, `learning_rate`,
  - `subsample`, `colsample_bytree`,
  - `reg_lambda`.
- For each trial:
  - We build an XGBoost model with those hyperparameters.
  - Optionally use **Stratified K-Fold** on the full factorial data (`USE_KFOLD=True`).
  - Evaluate on the **validation set** using the chosen primary metric (e.g. RMSE).
- Optuna chooses the best configuration.
- We retrain a final XGBoost model with these best hyperparameters and report ME, MAE, MSE, RMSE, MAPE, and R² on the validation set.

What to explore:

- Vary `N_TRIALS` to see how search budget affects the result.
- Adjust the **search ranges** for depth, learning rate, and regularization and see where Optuna converges.
- Compare the **best Optuna XGBoost** to:
  - the **grid-search XGBoost model**, and
  - the **neural network surrogates**.


In [ ]:
# CELL 9: XGBoost with Optuna hyperparameter search


# Choose scalers for inputs and target
# All options: "standard", "minmax", "robust", "log", "none"
X_SCALER_NAME = "standard"
Y_SCALER_NAME = "log"

# Choose primary optimization metric for Optuna
# Options: "mae", "mse", "rmse", "mape"
PRIMARY_METRIC_NAME = "rmse"
metric_fn = ERROR_FUNCTIONS[PRIMARY_METRIC_NAME]

# KFold configuration for tuning
USE_KFOLD = True
N_SPLITS_KFOLD = 5
N_BINS_STRAT = 6   # bins for stratified KFold on y

# Optuna configuration
N_TRIALS = 20      # reduce if needed
TIMEOUT = None     # or set seconds for a hard time limit

SAVE_MODEL = True
RUN_TAG = "my_model" # add a tag to identify your model bundle file

print("\n=== XGBoost Optuna search ===")
print("X scaler:", X_SCALER_NAME, " Y scaler:", Y_SCALER_NAME)
print("Primary metric:", PRIMARY_METRIC_NAME)
print("Use KFold:", USE_KFOLD)
print("N_TRIALS:", N_TRIALS)

def evaluate_xgb_config(params, use_kfold):
    """
    Evaluate one XGBoost hyperparameter configuration.

    Model-selection logic
    ---------------------
    If use_kfold is False:
        - Fit candidate model on X_train, y_train.
        - Evaluate candidate model on X_val, y_val.

    If use_kfold is True:
        - Use X_full, y_full as development data.
        - Rotate validation folds using StratifiedKFold.
        - Report mean/std metrics over folds.

    Returns
    -------
    score : float
        Value of PRIMARY_METRIC_NAME used for selecting the best configuration.
        Lower is better.
    details : dict
        Validation/CV metrics and fold information.
    """
    if use_kfold:
        # Stratified KFold on the full development data
        y_bins = build_stratified_bins_for_cv(y_full, nbins=N_BINS_STRAT)
        skf = StratifiedKFold(
            n_splits=N_SPLITS_KFOLD,
            shuffle=True,
            random_state=RANDOM_STATE,
        )

        fold_metric_rows = []

        for fold_idx, (train_idx, val_idx) in enumerate(skf.split(X_full, y_bins), start=1):
            X_tr = X_full[train_idx]
            y_tr = y_full[train_idx]
            X_va = X_full[val_idx]
            y_va = y_full[val_idx]

            # Create fresh scalers for each fold.
            # Fit scalers only on the training fold.
            x_scaler = create_scaler(X_SCALER_NAME)
            y_scaler = create_scaler(Y_SCALER_NAME)

            if x_scaler is not None:
                X_tr_scaled = x_scaler.fit_transform(X_tr)
                X_va_scaled = x_scaler.transform(X_va)
            else:
                X_tr_scaled = X_tr
                X_va_scaled = X_va

            if y_scaler is not None:
                y_tr_scaled = y_scaler.fit_transform(
                    y_tr.reshape(-1, 1)
                ).ravel()
            else:
                y_tr_scaled = y_tr

            model = xgb.XGBRegressor(
                n_estimators=params["n_estimators"],
                max_depth=params["max_depth"],
                learning_rate=params["learning_rate"],
                subsample=params["subsample"],
                colsample_bytree=params["colsample_bytree"],
                reg_lambda=params["reg_lambda"],
                objective="reg:squarederror",
                random_state=RANDOM_STATE,
            )

            model.fit(X_tr_scaled, y_tr_scaled)

            y_va_pred_scaled = model.predict(X_va_scaled)

            if y_scaler is not None:
                y_va_pred = y_scaler.inverse_transform(
                    y_va_pred_scaled.reshape(-1, 1)
                ).ravel()
            else:
                y_va_pred = y_va_pred_scaled

            fold_metrics = compute_all_metrics(y_va, y_va_pred)
            fold_metrics["fold"] = fold_idx
            fold_metric_rows.append(fold_metrics)

        df_folds = pd.DataFrame(fold_metric_rows)

        mean_metrics = {
            f"{metric}_mean": float(df_folds[metric].mean())
            for metric in ["ME", "MAE", "MSE", "RMSE", "MAPE", "R2"]
        }

        std_metrics = {
            f"{metric}_std": float(df_folds[metric].std())
            for metric in ["ME", "MAE", "MSE", "RMSE", "MAPE", "R2"]
        }

        score = mean_metrics[f"{PRIMARY_METRIC_NAME.upper()}_mean"]

        details = {
            "selection_type": "kfold_cv",
            "fold_metrics": df_folds,
        }
        details.update(mean_metrics)
        details.update(std_metrics)

        return score, details

    else:
        # Fixed validation strategy:
        # fit candidate on training data, evaluate candidate on validation data
        X_tr, y_tr = X_train, y_train
        X_va, y_va = X_val, y_val

        x_scaler = create_scaler(X_SCALER_NAME)
        y_scaler = create_scaler(Y_SCALER_NAME)

        if x_scaler is not None:
            X_tr_scaled = x_scaler.fit_transform(X_tr)
            X_va_scaled = x_scaler.transform(X_va)
        else:
            X_tr_scaled = X_tr
            X_va_scaled = X_va

        if y_scaler is not None:
            y_tr_scaled = y_scaler.fit_transform(
                y_tr.reshape(-1, 1)
            ).ravel()
        else:
            y_tr_scaled = y_tr

        model = xgb.XGBRegressor(
            n_estimators=params["n_estimators"],
            max_depth=params["max_depth"],
            learning_rate=params["learning_rate"],
            subsample=params["subsample"],
            colsample_bytree=params["colsample_bytree"],
            reg_lambda=params["reg_lambda"],
            objective="reg:squarederror",
            random_state=RANDOM_STATE,
        )

        model.fit(X_tr_scaled, y_tr_scaled)

        y_va_pred_scaled = model.predict(X_va_scaled)

        if y_scaler is not None:
            y_va_pred = y_scaler.inverse_transform(
                y_va_pred_scaled.reshape(-1, 1)
            ).ravel()
        else:
            y_va_pred = y_va_pred_scaled

        val_metrics = compute_all_metrics(y_va, y_va_pred)
        score = val_metrics[PRIMARY_METRIC_NAME.upper()]

        details = {
            "selection_type": "fixed_validation",
        }
        details.update(val_metrics)

        return score, details

def objective_xgb(trial):
    """
    Optuna objective for XGBoost.

    Samples hyperparameters, evaluates them with evaluate_xgb_config,
    and returns the chosen error metric (to be minimized).
    """
    params = {}

    # Number of trees
    params["n_estimators"] = trial.suggest_int("n_estimators", 50, 300, step=50)

    # Tree depth
    params["max_depth"] = trial.suggest_int("max_depth", 2, 6)

    # Learning rate
    params["learning_rate"] = trial.suggest_float(
        "learning_rate", 0.03, 0.3, log=True
    )

    # Subsampling fractions
    params["subsample"] = trial.suggest_float("subsample", 0.7, 1.0)
    params["colsample_bytree"] = trial.suggest_float("colsample_bytree", 0.7, 1.0)

    # L2 regularization
    params["reg_lambda"] = trial.suggest_float("reg_lambda", 1e-3, 1.0, log=True)

    score, details = evaluate_xgb_config(params, USE_KFOLD)
    trial.set_user_attr("details", details)

    return score


study_xgb = optuna.create_study(direction="minimize")
study_xgb.optimize(
    objective_xgb,
    n_trials=N_TRIALS,
    timeout=TIMEOUT,
    show_progress_bar=False,
)
# ------------------------------------------------------------
# Collect and display Optuna validation/CV results
# ------------------------------------------------------------
# Each Optuna trial is one candidate XGBoost setup.
# The score used by Optuna is the validation/CV score from evaluate_xgb_config().
# The untouched test set is not used here.

trial_rows_xgb = []
all_fold_metrics_xgb_optuna = []   # only used when USE_KFOLD=True

for trial in study_xgb.trials:
    # Skip failed or pruned trials, if any
    if trial.value is None:
        continue

    details = trial.user_attrs.get("details", {})

    row = {
        "trial_number": trial.number,
        "selection_score": trial.value,
        "primary_metric": PRIMARY_METRIC_NAME,
        "selection_type": details.get("selection_type", None),
    }

    # Add Optuna hyperparameters as separate columns
    for p_name, p_value in trial.params.items():
        row[p_name] = p_value

    # Add scalar validation/CV metrics
    # Do not add fold_metrics directly as a table column because it is hard to read.
    for key, value in details.items():
        if key not in ["fold_metrics", "selection_type"]:
            row[key] = value

    # Keep fold metrics separately for visible inspection
    if USE_KFOLD and "fold_metrics" in details:
        df_trial_folds = details["fold_metrics"].copy()
        df_trial_folds.insert(0, "trial_number", trial.number)
        df_trial_folds["params"] = str(trial.params)

        all_fold_metrics_xgb_optuna.append(df_trial_folds)

    trial_rows_xgb.append(row)


df_trials_xgb_optuna = pd.DataFrame(trial_rows_xgb).sort_values(
    "selection_score",
    ascending=True,
).reset_index(drop=True)

print("\nXGBoost Optuna validation/CV results")
print("One row per Optuna trial.")
print("Sorted by:", PRIMARY_METRIC_NAME)
display(df_trials_xgb_optuna)

# If K-fold was used, show all fold-by-fold metrics in a separate table.
if USE_KFOLD:
    df_xgb_optuna_fold_metrics = pd.concat(
        all_fold_metrics_xgb_optuna,
        ignore_index=True,
    )

    print("\nXGBoost Optuna fold-by-fold validation metrics")
    print("One row per trial and validation fold.")
    display(df_xgb_optuna_fold_metrics)
else:
    df_xgb_optuna_fold_metrics = None


print("\nOptuna best value (", PRIMARY_METRIC_NAME, "):", study_xgb.best_value)
print("Optuna best params:", study_xgb.best_params)

best_params_xgb = study_xgb.best_params

print("\nBest XGBoost Optuna setup:")
print("  Primary metric:", PRIMARY_METRIC_NAME)
print("  Selection score:", study_xgb.best_value)
print("  Best params:", best_params_xgb)


# ------------------------------------------------------------
# Retrain final XGBoost model on all development data
# ------------------------------------------------------------
# The validation/CV scores above were used only for model selection.
# After selecting the best hyperparameters, the final model is retrained
# on all development data: X_full, y_full.
#
# Final performance is evaluated later in Notebook 3 using the untouched test set.

X_tr_final, y_tr_final = X_full, y_full

x_scaler_final = create_scaler(X_SCALER_NAME)
y_scaler_final = create_scaler(Y_SCALER_NAME)

# Fit final scalers on all development data
if x_scaler_final is not None:
    X_tr_final_scaled = x_scaler_final.fit_transform(X_tr_final)
else:
    X_tr_final_scaled = X_tr_final

if y_scaler_final is not None:
    y_tr_final_scaled = y_scaler_final.fit_transform(
        y_tr_final.reshape(-1, 1)
    ).ravel()
else:
    y_tr_final_scaled = y_tr_final

best_xgb_model = xgb.XGBRegressor(
    n_estimators=best_params_xgb["n_estimators"],
    max_depth=best_params_xgb["max_depth"],
    learning_rate=best_params_xgb["learning_rate"],
    subsample=best_params_xgb["subsample"],
    colsample_bytree=best_params_xgb["colsample_bytree"],
    reg_lambda=best_params_xgb["reg_lambda"],
    objective="reg:squarederror",
    random_state=RANDOM_STATE,
)

best_xgb_model.fit(X_tr_final_scaled, y_tr_final_scaled)

print("\nFinal XGBoost Optuna model retrained on all development data.")
print("Use Notebook 3 to evaluate this saved model on the untouched test set and compare with the validation/CV results.")
# Optional: save the model bundle
if SAVE_MODEL:
    model_type = "xgb_optuna"
    path = build_student_model_path(SELECTED_REGIME, TARGET_COL, model_type, RUN_TAG)
    meta = {
    "regime": SELECTED_REGIME,
    "target": TARGET_COL,
    "model_type": model_type,
    "input_cols": INPUT_COLS,
    "x_scaler_name": X_SCALER_NAME,
    "y_scaler_name": Y_SCALER_NAME,
    "primary_metric": PRIMARY_METRIC_NAME,
    "split_method": SPLIT_METHOD,
    "use_kfold": USE_KFOLD,
    "n_trials": N_TRIALS,
    "best_params": best_params_xgb,
    "best_value": study_xgb.best_value,
    "selection_type": df_trials_xgb_optuna.iloc[0]["selection_type"],
    "selection_score": float(df_trials_xgb_optuna.iloc[0]["selection_score"]),
    "selection_metrics": {
        key: df_trials_xgb_optuna.iloc[0][key]
        for key in df_trials_xgb_optuna.columns
        if key not in ["params"]
    },
    "final_training_data": "all_development_data_X_full_y_full",
    }
    save_model_bundle(path, best_xgb_model, x_scaler_final, y_scaler_final, meta)


### Gaussian Process Surrogate

In this section we train **Gaussian Process Regression (GPR)** models using scikit-learn.

**Key ideas:**
- A GP defines a **distribution over functions**, controlled by a **kernel** (covariance function).
- Kernels encode assumptions about the function shape:
  - **RBF**: very smooth
  - **Matérn**: less smooth / rougher (nu=1.5 or nu=2.5)
  - **Sums of kernels**: combination of different behaviours
- Each kernel is wrapped with a **ConstantKernel** (trainable signal amplitude) and a **WhiteKernel** (trainable noise).

**What happens in this cell:**
- We use the same inputs (`INPUT_COLS`) and target (`TARGET_COL`).
- We apply scalers chosen by `X_SCALER_NAME` and `Y_SCALER_NAME` (GPR often benefits from e.g. `minmax` on X and `log` or `minmax` on Y).
- For each kernel in `GPR_KERNEL_NAMES`:
  - We build a kernel with **per-dimension lengthscales**, a trainable amplitude, and a trainable noise term.
  - We fit a `GaussianProcessRegressor` on the **training set**.
  - Kernel hyperparameters (lengthscales, variance, noise) are optimized and restarted `N_RESTARTS` times to reduce sensitivity to local optima.
  - We evaluate on the **validation set** and compute all metrics (ME, MAE, MSE, RMSE, MAPE, R²) in original units.
- We compare kernels based on `PRIMARY_METRIC_NAME` (e.g. RMSE) and select the **best kernel**.
- For the best model we print the fitted kernel structure including **lengthscales**, **signal variance**, **noise level**, and **log-marginal-likelihood**.

**What to explore:**
- How do GPR metrics compare to NN and XGBoost for the same target?
- Look at the **lengthscales**: is the GP more sensitive to WS or TI for this channel?
- Try different scalers for Y (e.g. `log` vs `minmax`) and observe the effect on fit quality.
- The **noise level** (WhiteKernel) should be very small for deterministic HAWC2 simulations; large noise suggests the kernel cannot explain the data structure well.
- In Notebook 3, we will use the GPR model to visualize **uncertainty bands** (mean ± 2σ) along WS/TI sweeps.

In [5]:
# CELL 10: Gaussian Process Regression (GPR) with different kernels

# GPR note:
# The continuous kernel parameters, such as length scales and noise level,
# are optimized internally during fitting by maximizing the marginal likelihood.
# Here, the validation set is used to compare higher-level modelling choices,
# such as the kernel family and scaler choices.

# -----------------------------------------------------------------------------
# Config
# -----------------------------------------------------------------------------
X_SCALER_NAME       = "minmax"
Y_SCALER_NAME       = "minmax"
PRIMARY_METRIC_NAME = "rmse"
GPR_KERNEL_NAMES    = ["RBF", "Matern32", "Matern52", "RBF+Matern32", "RBF+Matern52"]
N_RESTARTS          = 2  # number of optimizer restarts for fitting GPR hyperparameters
SAVE_MODEL          = True
RUN_TAG             = "_rmse_minmax"
METRIC_KEYS         = ["ME", "MAE", "MSE", "RMSE", "MAPE", "R2"]

# -----------------------------------------------------------------------------
# Kernel Builder
# -----------------------------------------------------------------------------
_KERNELS = {
    "rbf":                  lambda ls: ConstantKernel(1.0) * RBF(length_scale=ls),
    "matern32":             lambda ls: ConstantKernel(1.0) * Matern(length_scale=ls, nu=1.5),
    "matern52":             lambda ls: ConstantKernel(1.0) * Matern(length_scale=ls, nu=2.5),
}

_KERNEL_FORMATS = {
    ConstantKernel:         lambda k: f"ConstantKernel variance = {k.constant_value:.6g}",
    RBF:                    lambda k: f"RBF length_scale = {k.length_scale}",
    Matern:                 lambda k: f"Matern(nu={k.nu}) length_scale = {k.length_scale}",
    WhiteKernel:            lambda k: f"WhiteKernel noise_level = {k.noise_level:.6g}",
}


def build_kernel(name: str, input_dim: int):
    """
    Build a composite sklearn kernel from a '+'-separated name string.
    Each component is wrapped with a ConstantKernel (trainable amplitude).
    A WhiteKernel is appended to model observation noise.
    """
    ls = np.ones(input_dim)
    try:
        parts = [_KERNELS[p.strip()](ls) for p in name.lower().split("+")]
    except KeyError as e:
        raise ValueError(f"Unknown kernel component: {e}") from None
    return sum(parts[1:], parts[0]) + WhiteKernel(noise_level=0.01, noise_level_bounds=(1e-8, 1e2))


# -----------------------------------------------------------------------------
# Hyperparameter Print
# -----------------------------------------------------------------------------

def print_gpr_hyperparameters(model: GaussianProcessRegressor, indent: int = 2):
    """Print fitted kernel parameters and log-marginal-likelihood recursively."""
    def _print(k, pad):
        if isinstance(k, (Sum, Product)):
            _print(k.k1, pad); _print(k.k2, pad)
        else:
            fmt = _KERNEL_FORMATS.get(type(k), lambda k: f"{type(k).__name__}: {k.get_params()}")
            print(f"{pad}{fmt(k)}")

    print("\nFitted kernel:")
    _print(model.kernel_, " " * indent)
    print(f"  log-likelihood = {model.log_marginal_likelihood():.4f}")

# -----------------------------------------------------------------------------
# Evaluation
# -----------------------------------------------------------------------------

def evaluate_gpr_kernel(kernel_name: str):
    """
    Evaluate one GPR kernel using the fixed train/validation split.

    Model-selection logic
    ---------------------
    - Fit candidate GPR on X_train, y_train.
    - Evaluate candidate GPR on X_val, y_val.
    - Return validation metrics in original physical units.

    The final selected GPR is retrained later on all development data:
    X_full, y_full.
    """
    x_sc = create_scaler(X_SCALER_NAME)
    y_sc = create_scaler(Y_SCALER_NAME)

    # Fit scalers only on training data
    X_tr = x_sc.fit_transform(X_train) if x_sc else X_train
    X_va = x_sc.transform(X_val)       if x_sc else X_val

    y_tr = (
        y_sc.fit_transform(y_train.reshape(-1, 1)).ravel()
        if y_sc else y_train.ravel()
    )

    model = GaussianProcessRegressor(
        kernel=build_kernel(kernel_name, X_tr.shape[1]),
        alpha=1e-10,          # numerical jitter for Cholesky stability
        normalize_y=False,
        n_restarts_optimizer=N_RESTARTS,
    )

    model.fit(
        X_tr.astype(np.float64),
        y_tr.astype(np.float64),
    )

    # Predict on validation data
    y_pred_sc = model.predict(X_va.astype(np.float64))

    if y_sc:
        y_pred = y_sc.inverse_transform(
            y_pred_sc.reshape(-1, 1)
        ).ravel()
    else:
        y_pred = y_pred_sc

    # Validation metrics in original physical units
    metrics = compute_all_metrics(y_val, y_pred)

    score = metrics[PRIMARY_METRIC_NAME.upper()]

    return score, metrics


# -----------------------------------------------------------------------------
# Kernel comparison: model selection using validation data only
# -----------------------------------------------------------------------------

print(
    f"\n=== GPR kernel comparison | "
    f"X: {X_SCALER_NAME} | Y: {Y_SCALER_NAME} | "
    f"Metric: {PRIMARY_METRIC_NAME} ==="
)

gpr_results = []

for kname in GPR_KERNEL_NAMES:
    score, metrics = evaluate_gpr_kernel(kname)

    row = {
        "kernel": kname,
        "selection_score": score,
        "primary_metric": PRIMARY_METRIC_NAME,
        "selection_type": "fixed_validation",
        **metrics,
    }

    gpr_results.append(row)

    print(f"\n--- {kname} ---")
    print(f"Selection score ({PRIMARY_METRIC_NAME.upper()}): {score:.6g}")
    print("  " + "  ".join(f"{k}: {metrics[k]:.4g}" for k in METRIC_KEYS))


df_gpr_results = (
    pd.DataFrame(gpr_results)
    .sort_values("selection_score", ascending=True)
    .reset_index(drop=True)
)

print("\nGPR validation results")
print("One row per candidate kernel.")
print("Sorted by:", PRIMARY_METRIC_NAME)
display(df_gpr_results)

best_row_gpr = df_gpr_results.iloc[0]
best_kernel = best_row_gpr["kernel"]

print(f"\nBest GPR kernel: {best_kernel}")
print(f"Selection score ({PRIMARY_METRIC_NAME.upper()}): {best_row_gpr['selection_score']:.6g}")


# -----------------------------------------------------------------------------
# Retrain final GPR model on all development data
# -----------------------------------------------------------------------------
# The validation scores above were used only for kernel selection.
# After selecting the best kernel, the final GPR is retrained on all
# development data: X_full, y_full.
#
# Final performance is evaluated later in Notebook 3 using the untouched test set.

X_tr_final, y_tr_final = X_full, y_full

x_scaler_final = create_scaler(X_SCALER_NAME)
y_scaler_final = create_scaler(Y_SCALER_NAME)

# Fit final scalers on all development data
if x_scaler_final is not None:
    X_tr_final_scaled = x_scaler_final.fit_transform(X_tr_final)
else:
    X_tr_final_scaled = X_tr_final

if y_scaler_final is not None:
    y_tr_final_scaled = y_scaler_final.fit_transform(
        y_tr_final.reshape(-1, 1)
    ).ravel()
else:
    y_tr_final_scaled = y_tr_final

best_gpr_model = GaussianProcessRegressor(
    kernel=build_kernel(best_kernel, X_tr_final_scaled.shape[1]),
    alpha=1e-10,
    normalize_y=False,
    n_restarts_optimizer=N_RESTARTS,
)

best_gpr_model.fit(
    X_tr_final_scaled.astype(np.float64),
    y_tr_final_scaled.astype(np.float64),
)

print("\nFinal GPR model retrained on all development data.")
print("Use Notebook 3 to evaluate this saved model on the untouched test set and compare with the validation results.")

print_gpr_hyperparameters(best_gpr_model)


# -----------------------------------------------------------------------------
# Save final GPR model
# -----------------------------------------------------------------------------

if SAVE_MODEL:
    model_type = f"gpr_{best_kernel}"
    path = build_student_model_path(
        SELECTED_REGIME,
        TARGET_COL,
        model_type,
        RUN_TAG,
    )

    meta = {
        "regime": SELECTED_REGIME,
        "target": TARGET_COL,
        "model_type": model_type,
        "input_cols": INPUT_COLS,
        "x_scaler_name": X_SCALER_NAME,
        "y_scaler_name": Y_SCALER_NAME,
        "primary_metric": PRIMARY_METRIC_NAME,
        "selection_type": "fixed_validation",
        "selection_score": float(best_row_gpr["selection_score"]),
        "selection_metrics": {
            key: best_row_gpr[key]
            for key in df_gpr_results.columns
            if key not in ["kernel"]
        },
        "best_kernel": best_kernel,
        "kernel_candidates": GPR_KERNEL_NAMES,
        "n_restarts_optimizer": N_RESTARTS,
        "final_training_data": "all_development_data_X_full_y_full",
    }

    save_model_bundle(
        path,
        best_gpr_model,
        x_scaler_final,
        y_scaler_final,
        meta,
    )

    print(f"Model saved: {path}")


=== GPR kernel comparison | X: minmax | Y: minmax | Metric: rmse ===

--- RBF ---
Selection score (RMSE): 4.8067
  ME: -0.2239  MAE: 2.705  MSE: 23.1  RMSE: 4.807  MAPE: 3.747  R2: 0.9973

--- Matern32 ---
Selection score (RMSE): 2.08271
  ME: -0.2616  MAE: 1.23  MSE: 4.338  RMSE: 2.083  MAPE: 1.495  R2: 0.9995

--- Matern52 ---
Selection score (RMSE): 2.22112
  ME: -0.1701  MAE: 1.289  MSE: 4.933  RMSE: 2.221  MAPE: 1.705  R2: 0.9994

--- RBF+Matern32 ---
Selection score (RMSE): 2.09545
  ME: -0.2789  MAE: 1.233  MSE: 4.391  RMSE: 2.095  MAPE: 1.485  R2: 0.9995

--- RBF+Matern52 ---
Selection score (RMSE): 2.12251
  ME: -0.2826  MAE: 1.251  MSE: 4.505  RMSE: 2.123  MAPE: 1.399  R2: 0.9995

GPR validation results
One row per candidate kernel.
Sorted by: rmse


c:\Users\vaspas\AppData\Local\anaconda3\envs\General\lib\site-packages\sklearn\gaussian_process\kernels.py:440: ConvergenceWarning: The optimal value found for dimension 0 of parameter k1__k1__k2__length_scale is close to the specified lower bound 1e-05. Decreasing the bound and calling fit again may find a better value.
  warnings.warn(


,kernel,selection_score,primary_metric,selection_type,ME,MAE,MSE,RMSE,MAPE,R2
0,Matern32,2.082707,rmse,fixed_validation,-0.261617,1.229593,4.337666,2.082707,1.495353,0.999484
1,RBF+Matern32,2.095449,rmse,fixed_validation,-0.278868,1.232643,4.390906,2.095449,1.485470,0.999478
2,RBF+Matern52,2.122505,rmse,fixed_validation,-0.282561,1.250718,4.505028,2.122505,1.398673,0.999464
3,Matern52,2.221122,rmse,fixed_validation,-0.170144,1.288579,4.933383,2.221122,1.705317,0.999413
4,RBF,4.806699,rmse,fixed_validation,-0.223909,2.704736,23.104355,4.806699,3.747408,0.997253



Best GPR kernel: Matern32
Selection score (RMSE): 2.08271

Final GPR model retrained on all development data.
Use Notebook 3 to evaluate this saved model on the untouched test set and compare with the validation results.

Fitted kernel:
  ConstantKernel variance = 0.288802
  Matern(nu=1.5) length_scale = [0.38794136 2.33828274]
  WhiteKernel noise_level = 5.43308e-07
  log-likelihood = 1040.3068
Saved model bundle to: student\Normal__MSBMZ_loc_DEL4__gpr_Matern32___rmse_minmax.pkl
Model saved: student\Normal__MSBMZ_loc_DEL4__gpr_Matern32___rmse_minmax.pkl
